In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
from PIL import Image
import os

# preprocessing data

In [ ]:
#create validation caption
with open("/kaggle/input/image-processing-thai-language-image-captioning/capgen_v1.0_val.json", "r", encoding="utf-8") as file:
    data_val = json.load(file)

# Convert JSON to a list of tuples (image_path, index, caption)
records = []
for image_path, captions in data_val.items():
    for idx, caption in enumerate(captions):
        records.append((image_path, idx, caption))

# Create DataFrame
caption_val = pd.DataFrame(records, columns=["image_path", "caption_index", "caption_text"])

In [ ]:
#create training caption
with open("/kaggle/input/image-processing-thai-language-image-captioning/capgen_v1.0_train.json", "r", encoding="utf-8") as file:
    data_train = json.load(file)

# Convert JSON to a list of tuples (image_path, index, caption)
records = []
for image_path, captions in data_train.items():
    for idx, caption in enumerate(captions):
        records.append((image_path, idx, caption))

# Create DataFrame
caption_train = pd.DataFrame(records, columns=["image_path", "caption_index", "caption_text"])

In [ ]:
#img = Image.open(image_path)
caption_train['from'] = caption_train["image_path"].apply(lambda x: x.split('/')[0])
caption_train['img_id'] = caption_train["image_path"].apply(lambda x: x.split('/')[-1])

In [ ]:
caption_train

In [ ]:
caption_train_sub = caption_train[caption_train['caption_index']==0]
caption_train_sub = caption_train[caption_train['from']=='ipu24']

In [ ]:
caption_train_sub = caption_train_sub.drop('caption_index', axis=1)
caption

In [ ]:
caption_train_sub['from'].value_counts()

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar

# Function to load a single image based on row info
def load_image(row):
    # Define the base path for each source type
    if row['from'] == 'ipu24':
        if row['image_path'].split('/')[2] == 'travel':
            img_path = os.path.join('/kaggle/input/image-processing-thai-language-image-captioning/train/train/travel', row['img_id'])
        elif row['image_path'].split('/')[2] == 'food':
            img_path = os.path.join('/kaggle/input/image-processing-thai-language-image-captioning/train/train/food', row['img_id'])
    elif row['from'] == 'coco':
        img_path = os.path.join('/kaggle/input/coco-2017-dataset/coco2017/train2017', row['img_id'])

    return img_path


# Function to load images in batches and merge them into the same dataframe with a progress bar for the entire dataset
def load_images_with_progress(df, batch_size=100):
    all_batches = []  # List to store all processed batches
    
    # Create a progress bar for the entire dataset
    with tqdm(total=len(df), desc="Loading Images", unit="image") as pbar:
        for start in range(0, len(df), batch_size):
            batch = df.iloc[start:start + batch_size].copy()  # Create a copy of the batch to avoid SettingWithCopyWarning
            batch.loc[:, "new_image_path"] = batch.apply(load_image, axis=1)  # Load images for the batch
            all_batches.append(batch)  # Append the batch to the list
            
            # Update the progress bar after each batch
            pbar.update(batch_size)
    
    # Concatenate all batches back into a single dataframe
    df = pd.concat(all_batches, ignore_index=True)
    return df

In [ ]:
train_real = load_images_with_progress(caption_train_sub, batch_size = 100)

In [ ]:
train_real.to_csv("train_real_with_path.csv")

In [ ]:
train_real.head()

In [ ]:
train_real_fr = train_real.drop(['image_path', 'from', 'img_id',], axis=1)
train_real_fr

# Train the model

In [ ]:
instruction = "คิดแคปชั่นรูปนี้ให้หน่อย"
def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : Image.open(sample["new_image_path"])} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["caption_text"]} ]
        },
    ]
    return { "messages" : conversation }

In [ ]:
converted_dataset = []
for index, row in tqdm(train_real_fr.iterrows()):
    converted_dataset.append(convert_to_conversation(row))
#converted_dataset = [convert_to_conversation(sample) for sample in train_real]

# real training

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    # "scb10x/typhoon2-qwen2vl-7b-vision-instruct",
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

In [ ]:
model

In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

In [ ]:
trainer_stats = trainer.train()

# Process the test set

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

instruction = "คิดแคปชั่นรูปนี้ให้หน่อย"
test_cap = []
def generateCaption(image_path):
    image = Image.open(image_path)
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")
    
    output = model.generate(**inputs, max_new_tokens = 100,
                       use_cache = True, temperature = 0.5, min_p = 0.1)
    caption = tokenizer.decode(output[0], skip_special_tokens=True)
    test_cap.append(caption)

In [ ]:
folder_path = '/kaggle/input/image-processing-thai-language-image-captioning/test/test'

image_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
            image_paths.append(os.path.join(root, file))

print(len(image_paths))


In [ ]:
for img_path in tqdm(image_paths):
    generateCaption(img_path)

In [ ]:
test_cap[0]

In [ ]:
image_id = [k.split('/')[-1] for k in image_paths]

In [ ]:
image_id_real = [k.split('.')[0] for k in image_id]

In [ ]:
data = {'image_id': image_id_real, 'cc': test_cap}

# Create a DataFrame
submission = pd.DataFrame(data)

In [ ]:
submission.to_csv("test_ans_1.csv")

In [ ]:
submission['cc'] = submission['cc'].apply(lambda x: x.split('\n')[-1])


In [ ]:
submission['cc'] = submission['cc'].apply(lambda x: x.replace('"',''))

In [ ]:
submission

In [ ]:
submission.to_csv("first_sub1.csv")

# align labels

In [ ]:
sample = pd.read_csv('/kaggle/input/image-processing-thai-language-image-captioning/sample_submission.csv', dtype={'image_id': str})

In [ ]:
sampleid = sample['image_id']


In [ ]:
id_list = sampleid.to_list()

In [ ]:
id_list[:5]

In [ ]:
submission.head()

In [ ]:
df_final = submission.set_index('image_id').loc[id_list].reset_index()

In [ ]:
df_final.head(30)

In [ ]:
df_final = df_final.rename(columns={'cc': 'caption'})

In [ ]:
df_final

In [ ]:
df_final.to_csv('output.csv', index=False)

In [ ]:
pd.read_csv('/kaggle/working/output.csv')